# Object Detection Pipeline with StatLang

This notebook demonstrates a complete **object detection** workflow written
entirely in StatLang — from data generation, through model training, to
serving predictions on held-out images.

### What we will cover

| Step | Description | Key StatLang Feature |
|------|-------------|---------------------|
| 1 | Configure pipeline parameters | `%LET` macro variables |
| 2 | Generate synthetic training data | `PROC CVISION mode=generate_samples` |
| 3 | Explore the annotation dataset | `PROC FREQ`, `PROC MEANS`, `PROC PRINT` |
| 4 | Split into train / test sets | `DATA` step with filtering |
| 5 | Define reusable pipeline macros | `%MACRO` / `%MEND` |
| 6 | Train a Faster R-CNN detector | `PROC CVISION mode=train_detect` |
| 7 | Run detection on the test set | `PROC CVISION mode=serve` |
| 8 | Analyse detection quality | `PROC MEANS`, `PROC FREQ` |
| 9 | Export results | `PROC EXPORT` |

### Under the hood

- **Synthetic data**: coloured shapes (circles, rectangles, triangles)
  drawn on random backgrounds — each image ships with ground-truth
  bounding boxes.
- **Model**: torchvision Faster R-CNN with a ResNet-50 backbone,
  fine-tuned on the generated shapes.
- **Model store**: the trained model is persisted in StatLang's
  in-memory model store so it can be served in later steps.

> All code below is pure StatLang syntax.  No Python cells required.

---
## 1 &mdash; Pipeline Configuration

We use `%LET` macro variables to centralise every tuneable parameter.
Changing a value here propagates through the entire pipeline via `&var`
substitution — no need to hunt through individual PROC calls.

In [ ]:
/* ============================================================ */
/*  Pipeline configuration — edit these values to experiment     */
/* ============================================================ */

/* Data generation */
%LET n_train    = 30;
%LET n_test     = 10;
%LET img_size   = 128;
%LET data_seed  = 42;
%LET image_dir  = ./examples/work/objdet_images;

/* Training hyper-parameters */
%LET train_epochs = 3;
%LET learn_rate   = 0.005;
%LET batch_size   = 4;
%LET backbone     = resnet50;

/* Inference */
%LET model_name  = shape_detector;
%LET conf_thresh = 0.25;

%PUT === StatLang Object Detection Pipeline ===;
%PUT Training images : &n_train;
%PUT Test images     : &n_test;
%PUT Epochs          : &train_epochs;
%PUT Learning rate   : &learn_rate;
%PUT Confidence      : &conf_thresh;

---
## 2 &mdash; Generate Synthetic Object Detection Data

`PROC CVISION mode=generate_samples` creates small PNG images
containing randomly placed **circles** (red), **rectangles** (blue),
and **triangles** (green) on pastel backgrounds.  Each shape is
recorded as a row with its bounding-box coordinates and class label.

The output dataset `annotations` contains one row **per object**
(an image with three shapes produces three rows).

In [ ]:
proc cvision mode=generate_samples out=annotations
     n_train=&n_train  n_test=&n_test
     img_size=&img_size seed=&data_seed
     outdir='&image_dir';
run;

---
## 3 &mdash; Explore the Annotation Data

Before training, inspect what we generated:

- **PROC PRINT** shows a few rows so we can eyeball the schema.
- **PROC FREQ** summarises class balance across splits.
- **PROC MEANS** checks bounding-box coordinate ranges.

In [ ]:
/* Preview first 15 annotations */
title "Sample Annotations";
proc print data=annotations;
    where obs <= 15;
run;

In [ ]:
/* Class distribution by split */
proc freq data=annotations;
    tables split * label;
run;

In [ ]:
/* Bounding-box coordinate ranges */
proc means data=annotations;
    var x_min y_min x_max y_max;
run;

---
## 4 &mdash; Split Annotations into Train / Test

The generated dataset already contains a `split` column.  We use
simple DATA step filters to separate the two subsets.

In [ ]:
/* Training annotations (multiple rows per image) */
data train_annot;
    set annotations;
    if split = 'train';
run;

/* Test annotations — used later for quality evaluation */
data test_annot;
    set annotations;
    if split = 'test';
run;

---
## 5 &mdash; Define Reusable Pipeline Macros

We wrap each pipeline stage in a `%MACRO` so the workflow can be
re-run with different parameters without duplicating code.

| Macro | Purpose |
|-------|---------|
| `%train_detector` | Trains Faster R-CNN on labelled annotations |
| `%score_images`   | Scores images with a saved model |
| `%analyse_results`| Summarises detection quality |
| `%detection_pipeline` | End-to-end: train &rarr; score &rarr; analyse |

In [ ]:
/* ============================================================ */
/*  Macro: train_detector                                       */
/*  Trains a Faster R-CNN model on bounding-box annotations.    */
/* ============================================================ */
%macro train_detector(ds, mname, ep, lr, bbone);
    proc cvision data=&ds mode=train_detect
         model_name=&mname epochs=&ep lr=&lr
         backbone=&bbone out=training_log;
        image image_path;
    run;
%mend;

/* ============================================================ */
/*  Macro: score_images                                         */
/*  Runs detection on a dataset using a saved model.            */
/* ============================================================ */
%macro score_images(ds, mname, conf, outds);
    proc cvision data=&ds mode=serve out=&outds
         model_name=&mname confidence=&conf;
        image image_path;
    run;
%mend;

/* ============================================================ */
/*  Macro: analyse_results                                      */
/*  Reports summary statistics on the detection output.         */
/* ============================================================ */
%macro analyse_results(ds);
    proc means data=&ds;
        var confidence;
    run;

    proc freq data=&ds;
        tables predicted_label;
    run;
%mend;

/* ============================================================ */
/*  Macro: detection_pipeline                                   */
/*  Full pipeline: train -> score -> analyse.                   */
/* ============================================================ */
%macro detection_pipeline(train_ds, test_ds, mname, ep, lr, bbone, conf);
    %train_detector(&train_ds, &mname, &ep, &lr, &bbone);
    %score_images(&test_ds, &mname, &conf, detections);
    %analyse_results(detections);
%mend;

---
## 6 &mdash; Train the Object Detection Model

We call `%train_detector` which invokes
`PROC CVISION mode=train_detect` under the hood.  This:

1. Loads a COCO-pretrained **Faster R-CNN** (ResNet-50 backbone).
2. Replaces the classification head for our 3 shape classes
   (+ background).
3. Fine-tunes for `&train_epochs` epochs.
4. Saves the trained weights into the **model store** under
   `&model_name`.

In [ ]:
%train_detector(train_annot, &model_name, &train_epochs, &learn_rate, &backbone);

---
## 7 &mdash; Serve Predictions on Test Images

Now we load the trained model from the store and run detection on
the held-out test annotations.  `%score_images` calls
`PROC CVISION mode=serve` which:

1. Reconstructs the Faster R-CNN architecture.
2. Loads the saved `state_dict` from the model store.
3. Runs inference on each unique `image_path`.
4. Returns a dataset of predicted bounding boxes with labels and
   confidence scores.

In [ ]:
%score_images(test_annot, &model_name, &conf_thresh, detections);

In [ ]:
/* Preview the raw detection results */
title "Detection Results (test set)";
proc print data=detections;
run;

---
## 8 &mdash; Analyse Detection Quality

We look at two things:

- **Confidence distribution** (`PROC MEANS`) — how certain is the
  model about each detection?
- **Label frequency** (`PROC FREQ`) — which classes were detected
  most often?

In [ ]:
%analyse_results(detections);

---
## 9 &mdash; Full Pipeline in One Call

Because every stage is wrapped in a macro, we can re-run the entire
pipeline — with different hyper-parameters — using a single
`%detection_pipeline` invocation.

Below we retrain with a higher learning rate and stricter confidence
threshold to illustrate the composability of the macro system.

In [ ]:
/* Re-run pipeline with different hyper-parameters */
%LET model_v2     = shape_detector_v2;
%LET epochs_v2    = 4;
%LET lr_v2        = 0.008;
%LET conf_v2      = 0.40;

%detection_pipeline(train_annot, test_annot, &model_v2, &epochs_v2, &lr_v2, &backbone, &conf_v2);

---
## 10 &mdash; Export Results

Finally, we export the detection results to CSV for downstream
consumption (dashboards, further analysis, etc.).

In [ ]:
proc export data=detections
     outfile='./examples/work/detections.csv'
     dbms=csv;
run;

%PUT Pipeline complete.  Results exported to ./examples/work/detections.csv;

---
## Summary

This notebook demonstrated a **complete object detection pipeline**
written entirely in StatLang:

```
%LET  ──>  PROC CVISION generate  ──>  DATA step split
  │                                          │
  └──  %MACRO pipeline  ──>  train  ──>  score  ──>  analyse
                               │
                          model store
                               │
                          PROC EXPORT
```

**Key takeaways:**

- `%LET` variables centralise configuration — change once, propagate
  everywhere.
- `%MACRO` / `%MEND` encapsulate reusable stages that compose into
  end-to-end pipelines.
- `PROC CVISION` handles data generation, model training, and
  inference through its `mode=` option.
- The **model store** persists trained weights so scoring can happen
  in a separate step (or session).
- Standard StatLang procs (`PROC FREQ`, `PROC MEANS`, `PROC PRINT`,
  `PROC EXPORT`) integrate seamlessly for exploration and reporting.